# models

> Data models and URL bundles for the transcription source selection step

In [ ]:
#| default_exp models

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import List, Dict, Optional, Any, Callable
from typing_extensions import TypedDict
from dataclasses import dataclass, field
from cjm_fasthtml_file_browser.routes.handlers import FileBrowserRouters

from fasthtml.common import APIRouter

## SelectedFile

Represents a single audio or video file selected for transcription.

In [ ]:
#| export
class SelectedFile(TypedDict, total=False):
    """A selected audio or video file."""

    path: str  # Absolute path to the file
    filename: str  # Basename of the file
    file_type: str  # "audio" or "video"
    size_bytes: int  # File size in bytes
    duration: Optional[float]  # Duration in seconds (if known)
    format: str  # File extension without dot (e.g. "mp3", "mp4")

## ExtractionResult

Tracks the audio extraction status for a video file.

In [ ]:
#| export
class ExtractionResult(TypedDict, total=False):
    """Audio extraction result for a video file."""

    status: str  # "pending", "extracting", "complete", "error"
    audio_path: Optional[str]  # Path to extracted audio (when complete)
    error: Optional[str]  # Error message (when error)
    job_id: Optional[str]  # FFmpeg plugin job ID

## SourceSelectState

TypedDict for the source selection step workflow state. Provides type safety for the state structure stored by `cjm-workflow-state`.

In [ ]:
#| export
class SourceSelectState(TypedDict, total=False):
    """State for the transcription source selection step."""

    selected_files: List[SelectedFile]  # Ordered list of selected files
    selected_folders: List[str]  # Folder paths toggled for bulk add/remove
    current_directory: str  # Current file browser directory path
    browser_state: Dict[str, Any]  # Serialized BrowserState from file-browser library
    extraction_results: Dict[str, ExtractionResult]  # video_path -> extraction result
    verified: bool  # Whether selection has been verified
    committed_audio_paths: List[str]  # Final audio paths for next step

## SourceSelectUrls

URL bundle for source selection route handlers. All fields default to empty strings for safe construction before routes are registered.

In [ ]:
#| export
@dataclass
class SourceSelectUrls:
    """URL bundle for transcription source selection routes."""

    # Selection operations
    remove: str = ""  # Remove file from selection
    reorder: str = ""  # Reorder selection (SortableJS)
    clear: str = ""  # Clear all selected files
    toggle_all: str = ""  # Toggle all media in current directory

    # Preview
    preview: str = ""  # Preview a file (render player)
    media_src: str = ""  # Serve a local media file for HTML5 players

    # Verify
    verify: str = ""  # Verify selection + trigger extraction
    extraction_status: str = ""  # Poll extraction status

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
#| export
def _no_op_restore(session_id: str) -> None:
    """Default no-op for restore_state."""
    pass

def _no_op_reset() -> None:
    """Default no-op for reset_state."""
    pass

@dataclass
class BrowserResult:
    """Return type from init_browser_router."""
    routers: FileBrowserRouters = None    # File browser routers bundle
    render_panel: Callable = None           # (selected_files?, selected_folders?, session_id?) -> rendered panel
    restore_state: Callable = field(default=_no_op_restore)  # (session_id) -> None, restore persisted state
    reset_state: Callable = field(default=_no_op_reset)  # () -> None, reset in-memory caches

@dataclass
class SourceSelectResult:
    """Return type from init_source_select_routers."""
    routers: List[APIRouter] = field(default_factory=list)  # All routers to register
    urls: "SourceSelectUrls" = None                         # URL bundle
    render_panel: Callable = None                           # Render fn for browser panel
    restore_state: Callable = field(default=_no_op_restore)  # (session_id) -> None, restore persisted state
    reset_state: Callable = field(default=_no_op_reset)  # () -> None, reset in-memory caches

## Router Result Types

Dataclass return types for router initialization functions. Using dataclasses instead of tuples makes future additions non-breaking.